In [ ]:
import os
import json
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from google.colab import drive

# ── 0. 掛載 Google Drive ──
drive.mount('/drive')

# 🎯 0.5 新增：讀取加權解答本
weight_json_path = '/drive/MyDrive/OVO-Bench/question_weights.json'
if os.path.exists(weight_json_path):
    with open(weight_json_path, 'r', encoding='utf-8') as f:
        weight_table = json.load(f)
    print(f"📊 成功載入權重解答本，共計 {len(weight_table)} 題評分標準。")
else:
    weight_table = {}
    print("⚠️ 警告：找不到權重解答本，將退回使用二元正確率 (0/1)。")

# ── 1. 重新讀取 Drive 中「所有」的 JSON 檔案 ──
results_dir = '/drive/MyDrive/OVO-Bench/ovo_RF_data_v8'
data = []

print("正在讀取資料庫...")
for filename in os.listdir(results_dir):
    if filename.endswith('.json'):
        filepath = os.path.join(results_dir, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            try:
                res = json.load(f)
                params = res.get('parameters', {})
                if not params: continue

                detail = res.get('detail', [{}])[0]
                q_type = detail.get('q_type', 'UNKNOWN')
                is_correct = res.get('correct', 0)
                llm_answer = str(detail.get('llm_answer', '')) # 🎯 抓取 LLM 的真實答案 (A/B/C/D)
                item_id_str = str(res.get('item_id'))

                # 🎯 核心轉換：將答案轉為連續分數
                if item_id_str in weight_table:
                    weighted_score = weight_table[item_id_str].get(llm_answer, 0.0)
                else:
                    # 如果這題剛好沒在解答本裡，就退回原本的 0 或 1
                    weighted_score = 1.0 if is_correct else 0.0

                row = {'item_id': item_id_str, 'q_type': q_type, 'weighted_score': weighted_score}
                row.update(params)
                data.append(row)
            except Exception as e:
                pass

df = pd.DataFrame(data)
print(f"✅ 資料載入成功！目前總資料量飆升至：{len(df)} 筆")
print("-" * 60)

# ── 2. 處理題型特徵 (One-Hot Encoding) ──
if len(df) > 0:
    df_processed = pd.get_dummies(df, columns=['q_type'], prefix='is')
    for col in df_processed.columns:
        if col.startswith('is_'):
            df_processed[col] = df_processed[col].astype(int)

    # ── 3. 準備特徵 (X) 與目標 (y) ──
    type_features = [c for c in df_processed.columns if c.startswith('is_') and c != 'weighted_score']
    # 🎯 把排除名單中的 is_correct 改成 weighted_score
    all_parameters = [col for col in df.columns if col not in ['item_id', 'q_type', 'weighted_score', 'is_correct']]

    features = all_parameters + type_features
    X = df_processed[features]
    y = df_processed['weighted_score'] # 🎯 目標正式切換為「加權分數」

    # ── 4. 重新訓練「全能型」代理模型 ──
    proxy_model_all = RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=2, random_state=42)
    proxy_model_all.fit(X, y)

    print("✅ 新一代【加權迴歸】代理模型訓練完畢！大腦已升級。")
    print("=" * 60)

    # ── 5. 印出見證奇蹟的【全局特徵重要性】排行 ──
    importances = proxy_model_all.feature_importances_
    feature_imp_df = pd.DataFrame({
        'Feature': features,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)

    print("🔍 代理模型眼中的【最新特徵重要性】排行：")
    for index, row in feature_imp_df.iterrows():
        if row['Importance'] > 0:
            print(f"  ⭐ {row['Feature']:<25}: {row['Importance']:.4f}")
        else:
            print(f"  -  {row['Feature']:<25}: {row['Importance']:.4f}")
else:
    print("❌ 警告：沒有讀取到任何資料，請檢查 results_dir 路徑是否正確！")

"""Optuna"""


In [ ]:
!pip install optuna -q

全部跑




In [ ]:
import optuna
import pandas as pd

# 關閉 Optuna 的冗長輸出，保持畫面乾淨
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 定義所有要優化的題型
all_q_types = ['action', 'attribute', 'spatial']

# 用來儲存所有題型的最佳結果
best_params_all = {}

print("🚀 開始為所有題型尋找專屬【最高加權得分】黃金參數...\n")

# 透過迴圈，依序處理每種題型
for target_q_type in all_q_types:

    def objective(trial):
        # 1. 讓 Optuna 尋找最佳解 (Magic 7)
        realtime_near_n = trial.suggest_int('REALTIME_NEAR_N', 1, 6)
        big_window_base_sigma = trial.suggest_float('BIG_WINDOW_BASE_SIGMA', 50.0, 150.0)
        dedup_thresh_d = trial.suggest_float('DEDUP_THRESH_D', 0.85, 0.99)
        stu_move_thresh = trial.suggest_float('STU_MOVE_THRESH', 0.005, 0.05)
        k_sigmoid = trial.suggest_float('K_SIGMOID', 2.0, 10.0)
        semantic_near_n = trial.suggest_int('SEMANTIC_NEAR_N', 1, 6)
        dedup_thresh_a = trial.suggest_float('DEDUP_THRESH_A', 0.85, 0.99)

        # 2. 建立一筆預測資料 (X)
        trial_data = {col: 0.0 for col in features}

        # 填入 Optuna 猜測的參數
        trial_data['REALTIME_NEAR_N'] = realtime_near_n
        trial_data['BIG_WINDOW_BASE_SIGMA'] = big_window_base_sigma
        trial_data['DEDUP_THRESH_D'] = dedup_thresh_d
        trial_data['STU_MOVE_THRESH'] = stu_move_thresh
        trial_data['K_SIGMOID'] = k_sigmoid
        trial_data['SEMANTIC_NEAR_N'] = semantic_near_n
        trial_data['DEDUP_THRESH_A'] = dedup_thresh_a

        # 填入未優化的預設參數
        trial_data['BIG_WINDOW_K'] = 1.0
        trial_data['BIG_WINDOW_EPS'] = 0.01
        trial_data['BIG_WINDOW_BASE_N'] = 60
        trial_data['MIN_INTERVAL_BIG'] = 0.5
        trial_data['MIN_INTERVAL_SMALL'] = 0.25
        trial_data['MAX_N_SMALL'] = 8
        trial_data['SEMANTIC_NEAR_INTERVAL'] = 0.25
        trial_data['REALTIME_NEAR_INTERVAL'] = 0.25
        trial_data['STU_AREA_THRESH'] = 0.10
        trial_data['CLIP_DEDUP_THRESH'] = 0.95
        trial_data['K'] = 0.3
        trial_data['ALPHA'] = 1.0
        trial_data['BETA'] = 1.0

        # 動態設定題型開關！
        trial_data['is_action'] = 1 if target_q_type == 'action' else 0
        trial_data['is_attribute'] = 1 if target_q_type == 'attribute' else 0
        trial_data['is_spatial'] = 1 if target_q_type == 'spatial' else 0

        X_trial = pd.DataFrame([trial_data])[features]
        predicted_score = proxy_model_all.predict(X_trial)[0]

        # 🎯 回傳預期加權得分
        return predicted_score

    # 為目前的題型建立一個全新的 Optuna 任務
    study = optuna.create_study(direction='maximize')
    print(f"⏳ 正在優化【{target_q_type}】題型 (執行 2000 次模擬)...")
    study.optimize(objective, n_trials=2000)

    # 將最佳結果存起來
    best_params_all[target_q_type] = {
        'score': study.best_value,
        'params': study.best_params
    }
    print(f"   ✅ 【{target_q_type}】最高預期得分: {study.best_value:.4f}\n")

# ── 輸出華麗的總結報告 ──
print("=" * 60)
print("🏆 全題型【加權計分版】黃金參數總結報告 🏆")
print("=" * 60)
for q_type, result in best_params_all.items():
    print(f"🎯 題型: {q_type.upper()} (預期平均得分: {result['score']:.4f})")
    for k, v in result['params'].items():
        if isinstance(v, float):
            print(f"    {k}: {v:.4f}")
        else:
            print(f"    {k}: {v}")
    print("-" * 40)